In [1]:
import requests
from requests.structures import CaseInsensitiveDict
import numpy_financial as npf
from datetime import datetime as dt
import pandas as pd

headers = CaseInsensitiveDict()
headers["accept"] = "application/json"

In [2]:
class Portfolio:
    __baseUrlApi = "https://fintual.cl/api/real_assets/"

    def __init__(self, id):
        urlAssetInfo         = f"{self.__baseUrlApi}{id}"
        assetInfo            = requests.get(urlAssetInfo,  headers=headers).json()['data']['attributes']
        
        self.id          = id
        self.name        = assetInfo['name']
        self.startDate   = assetInfo['start_date']
        self.lastDate    = assetInfo['last_day']['date']
        self.lastPrice   = assetInfo['last_day']["net_asset_value"]
        # self.df          = pd.DataFrame() 
        self.df          = self.get_all_days()
    
    def __str__(self):
        return self.name

    def get_all_days(self):
        urlAssetInfoDays     = f"{self.__baseUrlApi}{self.id}/days?to_date=2024-08-28"
        # urlAssetInfoDays     = f"{self.__baseUrlApi}{self.id}/days"
        assetInfoDays        = requests.get(urlAssetInfoDays,  headers=headers).json()['data']

        df = pd.DataFrame(assetInfoDays)

        df = pd.json_normalize(df['attributes'])[['date', 'price', 'shareholders', 'total_assets', 'total_net_assets', 'outstanding_shares']]

        df = df[1:-1]

        df.columns = ('fecha','precio','accionistas','activos_totales','activos_neto_totales','acciones_en_circulación')

        df['ano'] = df['fecha'].apply(lambda x: dt.strptime(x, "%Y-%m-%d").year)
        df['mes'] = df['fecha'].apply(lambda x: dt.strptime(x, "%Y-%m-%d").month)
        df['dia'] = df['fecha'].apply(lambda x: dt.strptime(x, "%Y-%m-%d").day)

        self.df = df[['fecha', 'ano', 'mes', 'dia', 'precio', 'accionistas', 'activos_totales', 'activos_neto_totales', 'acciones_en_circulación']]
        return self.df 

    
    def ordenar_bolsa(self):
        self.df = self.df [['fecha', 'ano', 'mes', 'dia', 'precio', 'accionistas', 'activos_totales', 'activos_neto_totales', 'acciones_en_circulación']]
        return self.df 
    
    

In [12]:
myPortfolio = [186,187,188, 15077]

port186 = Portfolio(186)
# port186.get_all_days()
dt186 = port186.df

port187 = Portfolio(187)
# port187.get_all_days()
dt187 = port187.df

port188 = Portfolio(188)
# port188.get_all_days()
dt188 = port188.df


In [3]:
def make_excel_file(pt_list=[186,187,188]):
    li_port = []
    for n in pt_list:
        li_port.append(Portfolio(n))

    with pd.ExcelWriter("precios_c.xlsx") as writer:
        for port_n in li_port:
            port_n.df.to_excel(writer, sheet_name=f"{port_n.name}") 


make_excel_file()

In [11]:
dt187

,fecha,ano,mes,dia,precio,accionistas,activos_totales,activos_neto_totales,acciones_en_circulación
1,2024-08-27,2024,8,27,2061.8448,61777.0,2.814686e+11,2.377411e+11,1.153050e+08
2,2024-08-26,2024,8,26,2059.1434,61744.0,2.749396e+11,2.374435e+11,1.153118e+08
3,2024-08-25,2024,8,25,2070.4374,61697.0,2.757447e+11,2.384304e+11,1.151595e+08
4,2024-08-24,2024,8,24,2070.4578,61697.0,2.757385e+11,2.384328e+11,1.151595e+08
5,2024-08-23,2024,8,23,2070.4783,61697.0,2.757322e+11,2.384352e+11,1.151595e+08
...,...,...,...,...,...,...,...,...,...
2384,2018-02-17,2018,2,17,1006.8194,2.0,2.056585e+06,2.056374e+06,2.042446e+03
2385,2018-02-16,2018,2,16,1006.7866,2.0,2.056585e+06,2.056307e+06,2.042446e+03
2386,2018-02-15,2018,2,15,1006.9736,2.0,2.056766e+06,2.056689e+06,2.042446e+03
2387,2018-02-14,2018,2,14,1005.8570,2.0,2.060960e+06,1.804686e+06,1.794177e+03


In [7]:

with pd.ExcelWriter("precios_c.xlsx") as writer:
    dt186.to_excel(writer, sheet_name=f"{port186.name}")  
    dt187.to_excel(writer, sheet_name=f"{port187.name}")  
    dt188.to_excel(writer, sheet_name=f"{port188.name}") 

In [8]:
class Portfolio:
    __baseUrlApi = "https://fintual.cl/api/real_assets/"

    def __init__(self, id):
        urlAssetInfo         = f"{self.__baseUrlApi}{id}"
        assetInfo            = requests.get(urlAssetInfo,  headers=headers).json()['data']['attributes']
        
        self.id          = id
        self.name        = assetInfo['name']
        self.startDate   = assetInfo['start_date']
        self.lastDate    = assetInfo['last_day']['date']
        self.lastPrice   = assetInfo['last_day']["net_asset_value"]
        self.df          = pd.DataFrame() 

    def get_all_days(self):
        urlAssetInfoDays     = f"{self.__baseUrlApi}{self.id}/days?to_date=2024-08-28"
        # urlAssetInfoDays     = f"{self.__baseUrlApi}{self.id}/days"
        assetInfoDays        = requests.get(urlAssetInfoDays,  headers=headers).json()['data']

        df = pd.DataFrame(assetInfoDays)

        df = pd.json_normalize(df['attributes'])[['date', 'price', 'shareholders', 'total_assets', 'total_net_assets', 'outstanding_shares']]

        if dt.now().hour < 19:
            df = df[1:-1]

        df.columns = ('fecha','precio','accionistas','activos_totales','activos_neto_totales','acciones_en_circulación')

        self.df = df.set_index("date")
        return self.df 

In [9]:
info = pd.read_excel(f"precios.xlsx",sheet_name='Risky Norris')

# Diccionario donde se guardarán los datos importantes del documento
datos_excel = {}

info

,Unnamed: 0,fecha,ano,mes,dia,precio,accionistas,activos_totales,activos_neto_totales,acciones_en_circulación
0,1,2024-08-27,2024,8,27,2636.8129,53591,322465959285,2.368288e+11,8.981631e+07
1,2,2024-08-26,2024,8,26,2631.3190,53606,324695617880,2.364713e+11,8.986797e+07
2,3,2024-08-25,2024,8,25,2661.7973,53590,325170749998,2.391945e+11,8.986201e+07
3,4,2024-08-24,2024,8,24,2661.8805,53590,325170342960,2.392019e+11,8.986201e+07
4,5,2024-08-23,2024,8,23,2661.9637,53590,325169935923,2.392094e+11,8.986201e+07
...,...,...,...,...,...,...,...,...,...,...
2383,2384,2018-02-17,2018,2,17,1016.2728,2,1133361,1.133247e+06,1.115101e+03
2384,2385,2018-02-16,2018,2,16,1016.2871,2,1133340,1.133263e+06,1.115101e+03
2385,2386,2018-02-15,2018,2,15,1016.1704,2,1133173,1.133133e+06,1.115101e+03
2386,2387,2018-02-14,2018,2,14,1013.2619,2,1291954,9.853050e+05,9.724088e+02


In [10]:
info = pd.read_excel(f"precios.xlsx")

# Diccionario donde se guardarán los datos importantes Wdel documento
datos_excel = {}

# Aislamos la primera columna para encontrar algunas palabras claves 
# primera_col = info.iloc[:,0]

# Buscamos la posición del campo "Contrato" dentro de la primera columna del documento. 
# Esta indice nos permitirá encontrar los otros datos  
# indice_contrato = info.index[primera_col == 'Contrato:'][0]
info

,Unnamed: 0,fecha,ano,mes,dia,precio,accionistas,activos_totales,activos_neto_totales,acciones_en_circulación
0,1,2024-08-27,2024,8,27,2636.8129,53591,322465959285,2.368288e+11,8.981631e+07
1,2,2024-08-26,2024,8,26,2631.3190,53606,324695617880,2.364713e+11,8.986797e+07
2,3,2024-08-25,2024,8,25,2661.7973,53590,325170749998,2.391945e+11,8.986201e+07
3,4,2024-08-24,2024,8,24,2661.8805,53590,325170342960,2.392019e+11,8.986201e+07
4,5,2024-08-23,2024,8,23,2661.9637,53590,325169935923,2.392094e+11,8.986201e+07
...,...,...,...,...,...,...,...,...,...,...
2383,2384,2018-02-17,2018,2,17,1016.2728,2,1133361,1.133247e+06,1.115101e+03
2384,2385,2018-02-16,2018,2,16,1016.2871,2,1133340,1.133263e+06,1.115101e+03
2385,2386,2018-02-15,2018,2,15,1016.1704,2,1133173,1.133133e+06,1.115101e+03
2386,2387,2018-02-14,2018,2,14,1013.2619,2,1291954,9.853050e+05,9.724088e+02




*** Risky Norris ***
- All time profit          : 161.5
- Profit between two dates : 82.29
- Annualized profit        : 
        2018: 4.15
        2019: 37.52
        2020: 23.09
        2021: 31.16
        2022: -26.48
        2023: 32.61
        2024: 16.0



*** Moderate Pitt ***
- All time profit          : 105.39
- Profit between two dates : 43.63
- Annualized profit        : 
        2018: 4.04
        2019: 19.08
        2020: 15.93
        2021: 17.95
        2022: -12.29
        2023: 22.48
        2024: 12.86



*** Conservative Clooney ***
- All time profit          : 46.31
- Profit between two dates : 10.96
- Annualized profit        : 
        2018: 1.45
        2019: 4.81
        2020: 5.88
        2021: 6.89
        2022: 1.06
        2023: 10.57
        2024: 8.81



In [11]:
info2018 = info[info['ano']==2018]
info2018
info2024 = info[info['ano']==2024]
info2024

,Unnamed: 0,fecha,ano,mes,dia,precio,accionistas,activos_totales,activos_neto_totales,acciones_en_circulación
0,1,2024-08-27,2024,8,27,2636.8129,53591,322465959285,2.368288e+11,8.981631e+07
1,2,2024-08-26,2024,8,26,2631.3190,53606,324695617880,2.364713e+11,8.986797e+07
2,3,2024-08-25,2024,8,25,2661.7973,53590,325170749998,2.391945e+11,8.986201e+07
3,4,2024-08-24,2024,8,24,2661.8805,53590,325170342960,2.392019e+11,8.986201e+07
4,5,2024-08-23,2024,8,23,2661.9637,53590,325169935923,2.392094e+11,8.986201e+07
...,...,...,...,...,...,...,...,...,...,...
235,236,2024-01-05,2024,1,5,2228.9001,46227,236707252076,1.710249e+11,7.673063e+07
236,237,2024-01-04,2024,1,4,2210.5055,46176,235982102273,1.694162e+11,7.664140e+07
237,238,2024-01-03,2024,1,3,2220.0664,46123,236295934553,1.697547e+11,7.646378e+07
238,239,2024-01-02,2024,1,2,2236.1554,46019,237019117190,1.704946e+11,7.624451e+07


-	Tir anual de cada bolsa
-	Tir inicio a fin
-	Tir de A a B
-	Cuartil 1, mediana, Cuartil 3, Min, Max, Varianza, Promedio, Desviación estardar
-	Datos comparativos por meses (Covit, Ucrania)

In [12]:
print(info2024.iloc[0])
print("------------------------"*5)
print(info2024.iloc[-1])

Unnamed: 0                              1
fecha                          2024-08-27
ano                                  2024
mes                                     8
dia                                    27
precio                          2636.8129
accionistas                         53591
activos_totales              322465959285
activos_neto_totales       236828803150.0
acciones_en_circulación     89816309.4643
Name: 0, dtype: object
------------------------------------------------------------------------------------------------------------------------
Unnamed: 0                            240
fecha                          2024-01-01
ano                                  2024
mes                                     1
dia                                     1
precio                          2262.9479
accionistas                         45864
activos_totales              240867357439
activos_neto_totales       172100155910.0
acciones_en_circulación     76051311.9746
Name: 239, dtype

In [13]:
print(info2018.iloc[0])
print("------------------------"*5)
print(info2018.iloc[-1])

Unnamed: 0                         2067
fecha                        2018-12-31
ano                                2018
mes                                  12
dia                                  31
precio                        1045.5367
accionistas                        1058
activos_totales              1299570399
activos_neto_totales       1271001212.0
acciones_en_circulación     1215644.761
Name: 2066, dtype: object
------------------------------------------------------------------------------------------------------------------------
Unnamed: 0                       2388
fecha                      2018-02-13
ano                              2018
mes                                 2
dia                                13
precio                      1003.8325
accionistas                         1
activos_totales                708357
activos_neto_totales         401533.0
acciones_en_circulación         400.0
Name: 2387, dtype: object


In [14]:
print(info2018.iloc[-1]['precio'])
print(info2024.iloc[0]['precio'])

1003.8325
2636.8129


In [15]:
npf.irr([info2018.iloc[-1]['precio']*-1,info2024.iloc[0]['precio']])*100

162.674589635223

In [16]:
npf.irr([info2018.iloc[-1]['precio']*-1,info2018.iloc[0]['precio']])*100

4.154497886848674

In [17]:
fecha1 = '2018-06-01'
fecha2 = '2018-12-31'

f1 = info[info['fecha']==fecha1]
# print(info[info['fecha']==fecha1])
print(f1)
print("---------"*3)
print(info[info['fecha']==fecha2]['fecha'])

      Unnamed: 0       fecha   ano  mes  dia     precio  accionistas  \
2279        2280  2018-06-01  2018    6    1  1075.4866          112   

      activos_totales  activos_neto_totales  acciones_en_circulación  
2279        169715686           169474155.0              157579.0544  
---------------------------
2066    2018-12-31
Name: fecha, dtype: object


In [18]:
n1 = info.index[info['fecha'] == fecha1].tolist()[0]
n2 = info.index[info['fecha'] == fecha2].tolist()[0]
print(n1)
print(n2)
print(type(info))

2279
2066
<class 'pandas.core.frame.DataFrame'>


In [19]:
info.iloc[n1]

Unnamed: 0                        2280
fecha                       2018-06-01
ano                               2018
mes                                  6
dia                                  1
precio                       1075.4866
accionistas                        112
activos_totales              169715686
activos_neto_totales       169474155.0
acciones_en_circulación    157579.0544
Name: 2279, dtype: object

In [20]:
# fecha1 = '2018-06-01'
# fecha2 = '2018-12-31'

dt1 = info.iloc[n2:n1+1]
dt1

,Unnamed: 0,fecha,ano,mes,dia,precio,accionistas,activos_totales,activos_neto_totales,acciones_en_circulación
2066,2067,2018-12-31,2018,12,31,1045.5367,1058,1299570399,1.271001e+09,1.215645e+06
2067,2068,2018-12-30,2018,12,30,1045.5708,1058,1299570399,1.271043e+09,1.215645e+06
2068,2069,2018-12-29,2018,12,29,1045.6048,1058,1299570399,1.271084e+09,1.215645e+06
2069,2070,2018-12-28,2018,12,28,1045.6389,1058,1299570399,1.271125e+09,1.215645e+06
2070,2071,2018-12-27,2018,12,27,1046.5598,1053,1293230705,1.263313e+09,1.207110e+06
...,...,...,...,...,...,...,...,...,...,...
2275,2276,2018-06-05,2018,6,5,1085.1324,117,234103640,1.797816e+08,1.656771e+05
2276,2277,2018-06-04,2018,6,4,1078.1765,115,171407163,1.711491e+08,1.587394e+05
2277,2278,2018-06-03,2018,6,3,1075.4164,112,169715686,1.694631e+08,1.575791e+05
2278,2279,2018-06-02,2018,6,2,1075.4515,112,169715686,1.694686e+08,1.575791e+05
